# Aerial Object Classification & Detection
## Notebook 4 — Streamlit Deployment

In this notebook I'm deploying the best model (EfficientNetB0) using Streamlit.
Since we are on Google Colab, I'm using ngrok to get a public URL for the app.

In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf

# loading the saved best model
model = tf.keras.models.load_model(
    '/content/drive/MyDrive/aerial_project/models/efficientnet_best.h5'
)
print(f'Input shape: {model.input_shape}')

Write Streamlit App


In [ ]:
%%writefile app.py

import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
from tensorflow.keras.applications.efficientnet import preprocess_input

st.set_page_config(
    page_title='Aerial Object Classifier',
    page_icon='🦅',
    layout='centered'
)

@st.cache_resource
def load_model():
    return tf.keras.models.load_model(
        '/content/drive/MyDrive/aerial_project/models/efficientnet_best.h5'
    )

model = load_model()

# sidebar info
st.sidebar.title('Model Info')
st.sidebar.markdown("""
**Model:** EfficientNetB0

**Performance:**
| Metric | Score |
|--------|-------|
| Accuracy | 97.73% |
| Precision | 97.85% |
| Recall | 96.81% |
| F1 Score | 97.33% |

**Classes:** Bird, Drone
""")

st.title('🦅 Aerial Object Classifier')
st.subheader('Bird vs Drone Detection')
st.markdown('Upload an aerial image to classify it as **Bird** or **Drone**')
st.divider()

uploaded_file = st.file_uploader('Upload Image', type=['jpg', 'jpeg', 'png'])

if uploaded_file is not None:
    img = Image.open(uploaded_file).convert('RGB')
    st.image(img, caption='Uploaded Image', use_column_width=True)
    st.divider()

    # preprocess same way as training
    img_resized = img.resize((224, 224))
    img_array   = np.array(img_resized, dtype=np.float32)
    img_array   = preprocess_input(img_array)
    img_array   = np.expand_dims(img_array, axis=0)

    with st.spinner('Analyzing image...'):
        prediction = model.predict(img_array, verbose=0)[0][0]

    bird_conf  = (1 - prediction) * 100
    drone_conf = prediction * 100

    if prediction < 0.5:
        label      = '🐦 Bird'
        confidence = bird_conf
        color      = 'green'
    else:
        label      = '🚁 Drone'
        confidence = drone_conf
        color      = 'red'

    st.markdown(f'## Prediction: :{color}[{label}]')
    st.progress(int(confidence) / 100)
    st.metric('Confidence Score', f'{confidence:.2f}%')
    st.divider()

    col1, col2 = st.columns(2)
    with col1:
        st.metric('🐦 Bird',  f'{bird_conf:.2f}%')
    with col2:
        st.metric('🚁 Drone', f'{drone_conf:.2f}%')

    st.divider()
    if confidence >= 90:
        st.success(f'High confidence detection: {label}')
    elif confidence >= 75:
        st.info(f'Good confidence: {label}')
    else:
        st.warning('Low confidence — try a clearer image')

else:
    st.info('Please upload an image to get started')


In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/aerial_project/', exist_ok=True)
shutil.copy('app.py', '/content/drive/MyDrive/aerial_project/app.py')

Creating requirements.txt

In [ ]:
%%writefile requirements.txt
tensorflow
streamlit
numpy
Pillow
scikit-learn
matplotlib
seaborn
pyngrok
opencv-python


In [ ]:
import shutil
shutil.copy('requirements.txt', '/content/drive/MyDrive/aerial_project/requirements.txt')
print('requirements.txt saved to Drive!')

Launch App with Ngrok

In [ ]:
from pyngrok import ngrok
import subprocess, time, os

os.system('pkill -f streamlit')
ngrok.kill()
time.sleep(2)

# add your ngrok token here
NGROK_TOKEN = 'PASTE_YOUR_TOKEN_HERE'
ngrok.set_auth_token(NGROK_TOKEN)

subprocess.Popen([
    'streamlit', 'run', 'app.py',
    '--server.port', '8501',
    '--server.headless', 'true',
    '--server.enableCORS', 'false'
])

time.sleep(4)
public_url = ngrok.connect(8501)

print('=' * 50)
print('APP IS LIVE!')
print(f'URL: {public_url}')
print('=' * 50)

 Keep App Running

In [ ]:
import time
print('App is running!')
print('Do not stop this cell while using the app')
while True:
    time.sleep(60)
    print('App still running...')